In [1]:
import pandas as pd
import numpy as np

In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from scipy import stats

class FootballMatchPredictor:
    def __init__(self, team_stats_df):
        """
        Initialize the predictor with team statistics DataFrame
        """
        self.team_stats = team_stats_df
        
        # Verify and clean column names
        self.feature_columns = [
            'goals_per90', 'assists_per90', 'non_pen_goals_per90',
            'xg_per90', 'xg_assist_per90', 'npxg_per90', 
            'progressive_carries_per_90', 'progressive_passes_per_90'
        ]
        
        # Verify all required columns exist
        missing_columns = [col for col in self.feature_columns if col not in self.team_stats.columns]
        if missing_columns:
            raise ValueError(f"Missing required columns: {missing_columns}")
            
        # Initialize scalers and models
        self.scaler = StandardScaler()
        self.outcome_model = RandomForestClassifier(n_estimators=100, random_state=42)
        self.goals_model = RandomForestRegressor(n_estimators=100, random_state=42)
        
        # Prepare the training data
        self._prepare_training_data()
        
    def _prepare_training_data(self):
        """
        Prepare training data by creating synthetic matches between teams
        """
        matches = []
        teams = self.team_stats['team'].values
        
        for home_team in teams:
            for away_team in teams:
                if home_team != away_team:
                    try:
                        home_stats = self.team_stats[self.team_stats['team'] == home_team][self.feature_columns].values[0]
                        away_stats = self.team_stats[self.team_stats['team'] == away_team][self.feature_columns].values[0]
                        
                        # Combine features for both teams
                        match_features = np.concatenate([home_stats, away_stats])
                        
                        # Generate synthetic outcome based on team stats
                        home_strength = np.mean(home_stats[:3])  # Using scoring metrics
                        away_strength = np.mean(away_stats[:3])
                        
                        # Calculate probabilities
                        home_advantage = 1.1  # Home team advantage multiplier
                        total_strength = (home_strength * home_advantage) + away_strength
                        home_win_prob = (home_strength * home_advantage) / total_strength
                        
                        # Determine outcome (0: Away Win, 1: Draw, 2: Home Win)
                        random_val = np.random.random()
                        if random_val < home_win_prob:
                            outcome = 2
                        elif random_val < home_win_prob + 0.25:  # 25% chance of draw
                            outcome = 1
                        else:
                            outcome = 0
                            
                        # Estimate total goals
                        expected_goals = (home_stats[3] + away_stats[3]) * 0.9  # Using xG
                        
                        matches.append({
                            'features': match_features,
                            'outcome': outcome,
                            'total_goals': expected_goals
                        })
                    except IndexError:
                        print(f"Warning: Could not process match between {home_team} and {away_team}")
                        continue
        
        if not matches:
            raise ValueError("No valid matches could be created from the provided data")
            
        # Prepare training arrays
        X = np.array([m['features'] for m in matches])
        self.y_outcome = np.array([m['outcome'] for m in matches])
        self.y_goals = np.array([m['total_goals'] for m in matches])
        
        # Scale features
        self.X_scaled = self.scaler.fit_transform(X)
        
        # Train models
        self.outcome_model.fit(self.X_scaled, self.y_outcome)
        self.goals_model.fit(self.X_scaled, self.y_goals)
    
    def predict_match(self, home_team, away_team):
        """
        Predict the outcome and total goals for a match between two teams
        """
        # Verify teams exist in dataset
        if home_team not in self.team_stats['team'].values:
            raise KeyError(f"Home team '{home_team}' not found in dataset")
        if away_team not in self.team_stats['team'].values:
            raise KeyError(f"Away team '{away_team}' not found in dataset")
            
        # Get team stats
        home_stats = self.team_stats[self.team_stats['team'] == home_team][self.feature_columns].values[0]
        away_stats = self.team_stats[self.team_stats['team'] == away_team][self.feature_columns].values[0]
        
        # Combine and scale features
        match_features = np.concatenate([home_stats, away_stats]).reshape(1, -1)
        scaled_features = self.scaler.transform(match_features)
        
        # Get predictions
        outcome_probs = self.outcome_model.predict_proba(scaled_features)[0]
        predicted_goals = self.goals_model.predict(scaled_features)[0]
        
        # Calculate over/under probabilities
        over_under_thresholds = [1.5, 2.5, 3.5]
        over_under_probs = {
            f"Over {threshold}": 1 - stats.norm.cdf(threshold, predicted_goals, 1.2)
            for threshold in over_under_thresholds
        }
        
        return {
            'home_win_prob': outcome_probs[2],
            'draw_prob': outcome_probs[1],
            'away_win_prob': outcome_probs[0],
            'predicted_goals': predicted_goals,
            'over_under_probabilities': over_under_probs
        }
    
    def predict_fixture_list(self, fixture_df):
        """
        Predict outcomes for a list of fixtures from a DataFrame
        """
        if 'home_team' not in fixture_df.columns or 'away_team' not in fixture_df.columns:
            raise ValueError("Fixture DataFrame must contain 'home_team' and 'away_team' columns")
            
        predictions = []
        
        for _, row in fixture_df.iterrows():
            home_team = row['home_team']
            away_team = row['away_team']
            
            try:
                prediction = self.predict_match(home_team, away_team)
                predictions.append({
                    'home_team': home_team,
                    'away_team': away_team,
                    **prediction
                })
            except KeyError as e:
                print(f"Warning: {str(e)}")
                continue
        
        if not predictions:
            raise ValueError("No valid predictions could be made from the provided fixtures")
            
        return pd.DataFrame(predictions)

# Example usage
def load_and_predict_fixtures(team_stats_file, fixture_file):
    """
    Load team stats and fixture list, then make predictions
    """
    # Load team stats
    team_stats = pd.read_csv(team_stats_file)
    
    # Create example fixtures if fixture file doesn't exist
    try:
        fixtures = pd.read_excel(fixture_file)
    except FileNotFoundError:
        print("Fixture file not found. Creating example fixtures...")
        fixtures = pd.DataFrame({
            'home_team': ['Arsenal', 'Liverpool', 'Manchester City'],
            'away_team': ['Manchester Utd', 'Everton', 'Tottenham']
        })
    
    # Initialize predictor
    predictor = FootballMatchPredictor(team_stats)
    
    # Make predictions
    predictions = predictor.predict_fixture_list(fixtures)
    
    return predictions

# Let's test it with some example fixtures
example_fixtures = pd.DataFrame({
    'home_team': ['Arsenal', 'Liverpool', 'Manchester City'],
    'away_team': ['Manchester Utd', 'Everton', 'Tottenham']
})
example_fixtures.to_excel('fixtures.xlsx', index=False)

# Now try the prediction
predictions = load_and_predict_fixtures('cleaned_data.csv', 'fixtures.xlsx')
print("\nPredictions:")
print(predictions)


Predictions:
         home_team       away_team  home_win_prob  draw_prob  away_win_prob  \
0          Arsenal  Manchester Utd           0.17       0.12           0.71   
1        Liverpool         Everton           0.97       0.03           0.00   
2  Manchester City       Tottenham           0.84       0.12           0.04   

   predicted_goals                           over_under_probabilities  
0          2.75454  {'Over 1.5': 0.8520924809763708, 'Over 2.5': 0...  
1          3.00690  {'Over 1.5': 0.8953966875722356, 'Over 2.5': 0...  
2          3.22551  {'Over 1.5': 0.9247723322788711, 'Over 2.5': 0...  


In [3]:
predictions = load_and_predict_fixtures('cleaned_data.csv', 'fixtures.xlsx')
print(predictions)

         home_team       away_team  home_win_prob  draw_prob  away_win_prob  \
0          Arsenal  Manchester Utd           0.89       0.08           0.03   
1        Liverpool         Everton           0.88       0.12           0.00   
2  Manchester City       Tottenham           0.05       0.82           0.13   

   predicted_goals                           over_under_probabilities  
0          2.75454  {'Over 1.5': 0.8520924809763708, 'Over 2.5': 0...  
1          3.00690  {'Over 1.5': 0.8953966875722356, 'Over 2.5': 0...  
2          3.22551  {'Over 1.5': 0.9247723322788711, 'Over 2.5': 0...  


In [4]:
team_stats = pd.read_csv('cleaned_data.csv')
print(team_stats['team'].unique())

['Alavés' 'Angers' 'Arsenal' 'Aston Villa' 'Atalanta' 'Athletic Club'
 'Atlético Madrid' 'Augsburg' 'Auxerre' 'Barcelona' 'Bayern Munich'
 'Betis' 'Bochum' 'Bologna' 'Bournemouth' 'Brentford' 'Brest' 'Brighton'
 'Cagliari' 'Celta Vigo' 'Chelsea' 'Como' 'Crystal Palace' 'Dortmund'
 'Eint Frankfurt' 'Empoli' 'Espanyol' 'Everton' 'Fiorentina' 'Freiburg'
 'Fulham' 'Genoa' 'Getafe' 'Girona' 'Gladbach' 'Heidenheim'
 'Hellas Verona' 'Hoffenheim' 'Holstein Kiel' 'Inter' 'Ipswich Town'
 'Juventus' 'Las Palmas' 'Lazio' 'Le Havre' 'Lecce' 'Leganés'
 'Leicester City' 'Lens' 'Leverkusen' 'Lille' 'Liverpool' 'Lyon'
 'Mainz 05' 'Mallorca' 'Manchester City' 'Manchester Utd' 'Marseille'
 'Milan' 'Monaco' 'Montpellier' 'Monza' 'Nantes' 'Napoli' 'Newcastle Utd'
 'Nice' "Nott'ham Forest" 'Osasuna' 'Paris S-G' 'Parma' 'Rayo Vallecano'
 'RB Leipzig' 'Real Madrid' 'Real Sociedad' 'Reims' 'Rennes' 'Roma'
 'Saint-Étienne' 'Sevilla' 'Southampton' 'St. Pauli' 'Strasbourg'
 'Stuttgart' 'Torino' 'Tottenham' 'Toulo